# 04_distortion_model — Cortical distortion model: inverse fitting and parameter selection

**Manuscript:** Results sections 4-5; Methods 'Cortical distortion model', 'Inverse fitting', 'Parameter selection'; Supplementary S11 (retinal-family model), S13 (stability), Figure S1, tab:modelfits, tab:fit_stability.

The two-component model delta_theta = beta_s cos(theta - 90) + beta_c cos(theta - theta_conf) is fitted on a 1,326-cell grid by minimising a z-scored composite of loss atoms: psychophysical JND (`behav_loss.py`), Delta-RDM in a PCA(6) space (`neural_loss.py`) and hV4 LOCO voxel prediction. `s10b_v6_pca_rdm.py` runs every loss combination over N = 300 control 5-train/2-test resamples and applies the three gates; `s17_hc_loo.py` and `s18_heldout_predictive.py` / `s19_allcandidate_heldout.py` give the strict 7-fold leave-one-control-out statistics; `s10b_v6_srm_rdm.py` repeats the procedure with the RDM atom in the SRM basis; `rc_1dof.py` + `machado_simulator.py` implement the retinal-family comparison model. The committed `s10b_*_summary_*.json` files are slimmed copies of the 13-87 MB originals (the `summary` block is verbatim; per-resample fits are kept for the reported combinations only).

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_04_distortion_model.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/s10b_v6_pca_rdm_summary_sub-0{8,9}.json` | `scripts/s10b_v6_pca_rdm.py` | every loss combination x model over N = 300 resamples, PCA basis (production); slimmed copy, `summary` verbatim |
| `results/s10b_v6_srm_rdm_summary_sub-0{8,9}.json` | `scripts/s10b_v6_srm_rdm.py` | same with the RDM atom in the SRM basis |
| `results/precondition_table.json` | `scripts/s10a_precondition.py` | Gate 1 separation d per atom and ROI |
| `results/s19_allcandidate_heldout.json, s18_heldout_predictive.json, s17_hc_loo_results.json` | `scripts/s19_allcandidate_heldout.py, s18_heldout_predictive.py, s17_hc_loo.py` | strict 7-fold held-out statistics and neural-term ablation (tab:fit_stability) |
| `results/beta_sign_three_arms.json` | `scripts/filter_robustness_arms.py` | selected combinations refitted on the head-motion-corrected pipeline (S13) |
| `scripts/two_comp.py, rc_1dof.py, s8_loo_train_test.py` | `(imported)` | grid definition, R+C gain range, Delta-lambda anchors |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

sys.path.insert(0, str(Path("scripts").resolve()))
from two_comp import BS_GRID, BC_GRID, THETA_CONF
from rc_1dof import G_MIN, G_MAX, G_STEP
from s8_loo_train_test import DELTA_LAMBDA_BY_FAMILY
S = {"deutan": J("s10b_v6_pca_rdm_summary_sub-08.json"), "protan": J("s10b_v6_pca_rdm_summary_sub-09.json")}
M = {"deutan": J("s10b_v6_srm_rdm_summary_sub-08.json"), "protan": J("s10b_v6_srm_rdm_summary_sub-09.json")}
SEL = {"deutan": "γOY|RDMV2|noLOCO", "protan": "γALL|RDMV1|noLOCO"}
NEXT = {"deutan": "γALL|RDMV1|noLOCO", "protan": "γGB|RDMV1|noLOCO"}
PSY = {"deutan": "γOY|RDM_|noLOCO", "protan": "γALL|RDM_|noLOCO"}
RDM = {"deutan": "γ_|RDMV2|noLOCO", "protan": "γ_|RDMV1|noLOCO"}
def two(j, combo): return j["summary"][combo]["per_model"]["2comp"]
def argmin(j, combo): ps = two(j, combo)["param_summary"]; return (ps["bs_median"], ps["bc_median"])
def iqr(j, combo): ps = two(j, combo)["param_summary"]; return (ps["bs_iqr"], ps["bc_iqr"])
s19 = {c["label"] + "|" + c["subject"]: c for c in J("s19_allcandidate_heldout.json")["candidates"]}
pre = J("precondition_table.json")

V.start("04_distortion_model")

### Model and grid (Methods 'Cortical distortion model', 'Inverse fitting')

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.01 | Methods grid | beta_s grid: 26 values over [0, 50] at 2 degrees | `(26, 0.0, 50.0, 2.0)` |
| 04.02 | Methods grid | beta_c grid: 51 values over [-50, 50] at 2 degrees | `(51, -50.0, 50.0, 2.0)` |
| 04.03 | Methods grid | grid cells | `1326` |
| 04.04 | Methods model | theta_conf protan = 16 degrees | `16.0` |
| 04.05 | Methods model | theta_conf deutan = 150 degrees | `150.0` |
| 04.06 | S11 / Methods grid | R+C gain g over [0, 3] in steps of 0.05 (61 values) | `(0.0, 3.0, 0.05, 61)` |
| 04.07 | S11 | Delta-lambda anchors deutan 6.0, 6.5, 8.0 nm | `[6.0, 6.5, 8.0]` |
| 04.08 | S11 | Delta-lambda anchors protan 1.5, 3.0, 10.0 nm | `[1.5, 3.0, 10.0]` |
| 04.09 | Methods selection | N = 300 control 5-train/2-test resamples | `(300, 5)` |

In [2]:
print(len(BS_GRID), len(BC_GRID), THETA_CONF, G_MIN, G_MAX, G_STEP, DELTA_LAMBDA_BY_FAMILY, S["deutan"]["meta"])
n_g = int(round((G_MAX - G_MIN) / G_STEP)) + 1
V.check('04.01', 'Methods grid | beta_s grid: 26 values over [0, 50] at 2 degrees', (len(BS_GRID), float(min(BS_GRID)), float(max(BS_GRID)), float(BS_GRID[1] - BS_GRID[0])), (26, 0.0, 50.0, 2.0), mode='eq')
V.check('04.02', 'Methods grid | beta_c grid: 51 values over [-50, 50] at 2 degrees', (len(BC_GRID), float(min(BC_GRID)), float(max(BC_GRID)), float(BC_GRID[1] - BC_GRID[0])), (51, -50.0, 50.0, 2.0), mode='eq')
V.check('04.03', 'Methods grid | grid cells', len(BS_GRID) * len(BC_GRID), 1326, mode='eq')
V.check('04.04', 'Methods model | theta_conf protan = 16 degrees', float(THETA_CONF["protan"]), 16.0, nd=1)
V.check('04.05', 'Methods model | theta_conf deutan = 150 degrees', float(THETA_CONF["deutan"]), 150.0, nd=1)
V.check('04.06', 'S11 / Methods grid | R+C gain g over [0, 3] in steps of 0.05 (61 values)', (float(G_MIN), float(G_MAX), float(G_STEP), n_g), (0.0, 3.0, 0.05, 61), mode='eq')
V.check('04.07', 'S11 | Delta-lambda anchors deutan 6.0, 6.5, 8.0 nm', sorted(float(x) for x in DELTA_LAMBDA_BY_FAMILY["deutan"].values()), [6.0, 6.5, 8.0], mode='eq')
V.check('04.08', 'S11 | Delta-lambda anchors protan 1.5, 3.0, 10.0 nm', sorted(float(x) for x in DELTA_LAMBDA_BY_FAMILY["protan"].values()), [1.5, 3.0, 10.0], mode='eq')
V.check('04.09', 'Methods selection | N = 300 control 5-train/2-test resamples', (S["deutan"]["meta"]["N_resamples"], S["deutan"]["meta"]["subset_size"]), (300, 5), mode='eq')

26 51 {'protan': 16.0, 'deutan': 150.0} 0.0 3.0 0.05 {'deutan': {'DPS_lit': 6.0, 'Boehm_mid': 8.0, 'JND_Lamb': 6.5}, 'protan': {'DPS_lit': 10.0, 'Boehm_low': 3.0, 'JND_Lamb': 1.5}} {'N_resamples': 300, 'subset_size': 5, 'seed_base': 42, 'combo_range': [None, None]}
[OK ] 04.01 Methods grid | beta_s grid: 26 values over [0, 50] at 2 degrees: produced=(26, 0, 50, 2)  reported=(26, 0, 50, 2)
[OK ] 04.02 Methods grid | beta_c grid: 51 values over [-50, 50] at 2 degrees: produced=(51, -50, 50, 2)  reported=(51, -50, 50, 2)
[OK ] 04.03 Methods grid | grid cells: produced=1326  reported=1326
[OK ] 04.04 Methods model | theta_conf protan = 16 degrees: produced=16  reported=16
[OK ] 04.05 Methods model | theta_conf deutan = 150 degrees: produced=150  reported=150
[OK ] 04.06 S11 / Methods grid | R+C gain g over [0, 3] in steps of 0.05 (61 values): produced=(0, 3, 0.05, 61)  reported=(0, 3, 0.05, 61)
[OK ] 04.07 S11 | Delta-lambda anchors deutan 6.0, 6.5, 8.0 nm: produced=(6, 6.5, 8)  reported=(

### Gates and the selected fits (Results section 4; Supplementary tab:modelfits)
Gate 2 rejects a combination when at least half of its resample solutions reach a grid edge. Held-out loss and IQR are medians over the 300 resamples.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.10 | Results §4 ¶2 | deutan combinations passing the boundary-saturation gate | `25` |
| 04.11 | Results §4 ¶2 | protan combinations passing | `4` |
| 04.12 | Results §4 ¶2 | every gate-passing protan combination with an RDM atom returns (2, +24) | `True` |
| 04.13 | Results §4 ¶1 | the LOCO family entered neither winning combination | `False` |
| 04.14 | tab:modelfits | deutan selected beta_s | `6.0` |
| 04.15 | tab:modelfits | deutan selected beta_c | `-42.0` |
| 04.16 | tab:modelfits | deutan selected L_test | `-2.36` |
| 04.17 | tab:modelfits | deutan selected IQR | `2.15` |
| 04.18 | tab:modelfits | deutan next-ranked beta_s | `38.0` |
| 04.19 | tab:modelfits | deutan next-ranked beta_c | `-10.0` |
| 04.20 | tab:modelfits | deutan next-ranked L_test | `-1.14` |
| 04.21 | tab:modelfits | deutan next-ranked IQR | `0.86` |
| 04.22 | tab:modelfits | protan selected beta_s | `2.0` |
| 04.23 | tab:modelfits | protan selected beta_c | `24.0` |
| 04.24 | tab:modelfits | protan selected L_test | `-1.54` |
| 04.25 | tab:modelfits | protan selected IQR | `1.42` |
| 04.26 | tab:modelfits | protan next-ranked returns the same estimate | `(2.0, 24.0)` |
| 04.27 | tab:modelfits | protan next-ranked L_test | `-1.52` |
| 04.28 | tab:modelfits | protan next-ranked IQR | `1.41` |
| 04.29 | Results §4 ¶2 | the deutan next-ranked candidate is beta_s-dominant | `True` |

In [3]:
def passing(j): return [k for k, v in j["summary"].items() if v["per_model"]["2comp"]["boundary_rate"] < 0.5]
n_pass = {k: len(passing(S[k])) for k in S}
protan_rdm_pass = {k: argmin(S["protan"], k) for k in passing(S["protan"]) if "RDMV" in k}
loco_in_sel = any("|LOCO" in SEL[k] for k in SEL)
for k in S:
    print(k, "selected", argmin(S[k], SEL[k]), two(S[k], SEL[k])["test_loss_median"], two(S[k], SEL[k])["test_loss_iqr"], "next", argmin(S[k], NEXT[k]), two(S[k], NEXT[k])["test_loss_median"])
print(n_pass, protan_rdm_pass)
V.check('04.10', 'Results §4 ¶2 | deutan combinations passing the boundary-saturation gate', n_pass["deutan"], 25, mode='eq')
V.check('04.11', 'Results §4 ¶2 | protan combinations passing', n_pass["protan"], 4, mode='eq')
V.check('04.12', 'Results §4 ¶2 | every gate-passing protan combination with an RDM atom returns (2, +24)', all(v == (2.0, 24.0) for v in protan_rdm_pass.values()) and len(protan_rdm_pass) > 0, True, mode='eq')
V.check('04.13', 'Results §4 ¶1 | the LOCO family entered neither winning combination', loco_in_sel, False, mode='eq')
V.check('04.14', 'tab:modelfits | deutan selected beta_s', argmin(S["deutan"], SEL["deutan"])[0], 6.0, nd=0)
V.check('04.15', 'tab:modelfits | deutan selected beta_c', argmin(S["deutan"], SEL["deutan"])[1], -42.0, nd=0)
V.check('04.16', 'tab:modelfits | deutan selected L_test', two(S["deutan"], SEL["deutan"])["test_loss_median"], -2.36, nd=2)
V.check('04.17', 'tab:modelfits | deutan selected IQR', two(S["deutan"], SEL["deutan"])["test_loss_iqr"], 2.15, nd=2)
V.check('04.18', 'tab:modelfits | deutan next-ranked beta_s', argmin(S["deutan"], NEXT["deutan"])[0], 38.0, nd=0)
V.check('04.19', 'tab:modelfits | deutan next-ranked beta_c', argmin(S["deutan"], NEXT["deutan"])[1], -10.0, nd=0)
V.check('04.20', 'tab:modelfits | deutan next-ranked L_test', two(S["deutan"], NEXT["deutan"])["test_loss_median"], -1.14, nd=2)
V.check('04.21', 'tab:modelfits | deutan next-ranked IQR', two(S["deutan"], NEXT["deutan"])["test_loss_iqr"], 0.86, nd=2)
V.check('04.22', 'tab:modelfits | protan selected beta_s', argmin(S["protan"], SEL["protan"])[0], 2.0, nd=0)
V.check('04.23', 'tab:modelfits | protan selected beta_c', argmin(S["protan"], SEL["protan"])[1], 24.0, nd=0)
V.check('04.24', 'tab:modelfits | protan selected L_test', two(S["protan"], SEL["protan"])["test_loss_median"], -1.54, nd=2)
V.check('04.25', 'tab:modelfits | protan selected IQR', two(S["protan"], SEL["protan"])["test_loss_iqr"], 1.42, nd=2)
V.check('04.26', 'tab:modelfits | protan next-ranked returns the same estimate', argmin(S["protan"], NEXT["protan"]), (2.0, 24.0), mode='eq')
V.check('04.27', 'tab:modelfits | protan next-ranked L_test', two(S["protan"], NEXT["protan"])["test_loss_median"], -1.52, nd=2)
V.check('04.28', 'tab:modelfits | protan next-ranked IQR', two(S["protan"], NEXT["protan"])["test_loss_iqr"], 1.41, nd=2)
V.check('04.29', 'Results §4 ¶2 | the deutan next-ranked candidate is beta_s-dominant', abs(argmin(S["deutan"], NEXT["deutan"])[0]) > abs(argmin(S["deutan"], NEXT["deutan"])[1]), True, mode='eq')

deutan selected (6.0, -42.0) -2.359316295724163 2.149687623669479 next (38.0, -10.0) -1.1374315087944904
protan selected (2.0, 24.0) -1.5390701698772489 1.416685834164569 next (2.0, 24.0) -1.5187992352040045
{'deutan': 25, 'protan': 4} {'γGB|RDMV1|noLOCO': (2.0, 24.0), 'γALL|RDMV1|noLOCO': (2.0, 24.0)}
[OK ] 04.10 Results §4 ¶2 | deutan combinations passing the boundary-saturation gate: produced=25  reported=25
[OK ] 04.11 Results §4 ¶2 | protan combinations passing: produced=4  reported=4
[OK ] 04.12 Results §4 ¶2 | every gate-passing protan combination with an RDM atom returns (2, +24): produced=True  reported=True
[OK ] 04.13 Results §4 ¶1 | the LOCO family entered neither winning combination: produced=False  reported=False
[OK ] 04.14 tab:modelfits | deutan selected beta_s: produced=6  reported=6
[OK ] 04.15 tab:modelfits | deutan selected beta_c: produced=-42  reported=-42
[OK ] 04.16 tab:modelfits | deutan selected L_test: produced=-2.359  reported=-2.36
[OK ] 04.17 tab:modelfits

### Retinal-family comparison model (Results section 4 ¶5; Supplementary S11, tab:modelfits)
R+C fits at three Delta-lambda anchors on the selected combination; the manuscript reports the anchor of lowest held-out loss.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.30 | S11 / tab:modelfits | deutan R+C: 100% of resamples saturate at the reported anchor (lowest held-out loss; the three anchors give 0.71-1.00) | `1.0` |
| 04.31 | S11 / tab:modelfits | deutan R+C gain at the boundary g = 3.0 | `3.0` |
| 04.32 | S11 ¶3 | protan R+C: 41% saturated at the best anchor | `41.0` |
| 04.33 | S11 ¶3 | protan R+C gain g = 2.95 | `2.95` |
| 04.34 | S11 ¶3 | protan R+C held-out loss -0.86 | `-0.86` |
| 04.35 | S11 ¶3 | protan saturation ranges from 0% to 100% across anchors | `(0.0, 1.0)` |

In [4]:
rc = {}
for k in S:
    pm = S[k]["summary"][SEL[k]]["per_model"]
    anchors = {a: v for a, v in pm.items() if a.startswith("rc_")}
    best = min(anchors, key=lambda a: anchors[a]["test_loss_median"])
    rc[k] = dict(best=best, boundary=anchors[best]["boundary_rate"], g=anchors[best]["param_summary"].get("g_median", anchors[best]["param_summary"].get("g")), L=anchors[best]["test_loss_median"],
                 all_boundary=[v["boundary_rate"] for v in anchors.values()])
    print(k, rc[k], anchors[best]["param_summary"])
V.check('04.30', 'S11 / tab:modelfits | deutan R+C: 100% of resamples saturate at the reported anchor (lowest held-out loss; the three anchors give 0.71-1.00)', rc["deutan"]["boundary"], 1.0, nd=2)
V.check('04.31', 'S11 / tab:modelfits | deutan R+C gain at the boundary g = 3.0', rc["deutan"]["g"], 3.0, nd=2)
V.check('04.32', 'S11 ¶3 | protan R+C: 41% saturated at the best anchor', rc["protan"]["boundary"] * 100, 41.0, nd=0)
V.check('04.33', 'S11 ¶3 | protan R+C gain g = 2.95', rc["protan"]["g"], 2.95, nd=2)
V.check('04.34', 'S11 ¶3 | protan R+C held-out loss -0.86', rc["protan"]["L"], -0.86, nd=2)
V.check('04.35', 'S11 ¶3 | protan saturation ranges from 0% to 100% across anchors', (min(rc["protan"]["all_boundary"]), max(rc["protan"]["all_boundary"])), (0.0, 1.0), mode='pair', nd=2)

deutan {'best': 'rc_JND_Lamb', 'boundary': 1.0, 'g': 3.0, 'L': 0.18459922817097577, 'all_boundary': [1.0, 0.7066666666666667, 1.0]} {'g_median': 3.0, 'g_iqr': 0.0}
protan {'best': 'rc_Boehm_low', 'boundary': 0.41333333333333333, 'g': 2.95, 'L': -0.858609039546967, 'all_boundary': [0.0, 0.41333333333333333, 1.0]} {'g_median': 2.95, 'g_iqr': 0.09999999999999964}
[OK ] 04.30 S11 / tab:modelfits | deutan R+C: 100% of resamples saturate at the reported anchor (lowest held-out loss; the three anchors give 0.71-1.00): produced=1  reported=1
[OK ] 04.31 S11 / tab:modelfits | deutan R+C gain at the boundary g = 3.0: produced=3  reported=3
[OK ] 04.32 S11 ¶3 | protan R+C: 41% saturated at the best anchor: produced=41.33  reported=41
[OK ] 04.33 S11 ¶3 | protan R+C gain g = 2.95: produced=2.95  reported=2.95
[OK ] 04.34 S11 ¶3 | protan R+C held-out loss -0.86: produced=-0.8586  reported=-0.86
[OK ] 04.35 S11 ¶3 | protan saturation ranges from 0% to 100% across anchors: produced=(0, 1)  reported=(

### Held-out generalization and the separation precondition (Supplementary tab:fit_stability)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.36 | tab:fit_stability | deutan separation d at V1 / V2 / V3 / hV4 | `[2.31, 1.94, 0.86, 2.19]` |
| 04.37 | tab:fit_stability | protan separation d at V1 / V2 / V3 / hV4 | `[0.81, -0.23, -0.48, -0.24]` |
| 04.38 | Results §4 ¶1 | RDM atom admissible at all four ROIs in deutan and V1 alone in protan (d >= +0.5) | `(4, ['V1'])` |
| 04.39 | tab:fit_stability | deutan grid percentile (%) | `4.6` |
| 04.40 | tab:fit_stability | protan grid percentile (%) | `8.1` |
| 04.41 | Results §4 ¶3 | both within the top 8% of the 1,326-cell grid | `True` |
| 04.42 | tab:fit_stability | deutan Delta-L psychophysical atom | `-13.85` |
| 04.43 | tab:fit_stability | deutan folds favouring it (of 7) | `5` |
| 04.44 | tab:fit_stability | protan Delta-L psychophysical atom | `0.01` |
| 04.45 | tab:fit_stability | protan folds favouring it | `3` |
| 04.46 | tab:fit_stability | deutan Delta-L RDM atom | `-0.41` |
| 04.47 | tab:fit_stability | deutan RDM folds (7 of 7) | `7` |
| 04.48 | tab:fit_stability | protan Delta-L RDM atom | `-0.47` |
| 04.49 | tab:fit_stability | protan RDM folds (7 of 7) | `7` |

In [5]:
c8 = s19[SEL["deutan"] + "|sub-08"]; c9 = s19[SEL["protan"] + "|sub-09"]
d_sep = {k: [pre[r]["cohens_d"][s]["L_RDM"]["d"] for r in ("V1", "V2", "V3", "V4")] for k, s in (("deutan", "sub-08"), ("protan", "sub-09"))}
print(d_sep, c8["rdm_pct_med"], c9["rdm_pct_med"])
V.check('04.36', 'tab:fit_stability | deutan separation d at V1 / V2 / V3 / hV4', [round(x, 2) for x in d_sep["deutan"]], [2.31, 1.94, 0.86, 2.19], mode='list', nd=2)
V.check('04.37', 'tab:fit_stability | protan separation d at V1 / V2 / V3 / hV4', [round(x, 2) for x in d_sep["protan"]], [0.81, -0.23, -0.48, -0.24], mode='list', nd=2)
V.check('04.38', 'Results §4 ¶1 | RDM atom admissible at all four ROIs in deutan and V1 alone in protan (d >= +0.5)', (sum(d >= 0.5 for d in d_sep["deutan"]), [r for r, d in zip(ROIS, d_sep["protan"]) if d >= 0.5]), (4, ['V1']), mode='eq')
V.check('04.39', 'tab:fit_stability | deutan grid percentile (%)', c8["rdm_pct_med"] * 100, 4.6, nd=1)
V.check('04.40', 'tab:fit_stability | protan grid percentile (%)', c9["rdm_pct_med"] * 100, 8.1, nd=1)
V.check('04.41', 'Results §4 ¶3 | both within the top 8% of the 1,326-cell grid', max(c8["rdm_pct_med"], c9["rdm_pct_med"]) <= 0.081, True, mode='eq')
V.check('04.42', 'tab:fit_stability | deutan Delta-L psychophysical atom', c8["gamma_dL_med"], -13.85, nd=2)
V.check('04.43', 'tab:fit_stability | deutan folds favouring it (of 7)', round(c8["gamma_folds_beat00"] * 7), 5, mode='eq')
V.check('04.44', 'tab:fit_stability | protan Delta-L psychophysical atom', c9["gamma_dL_med"], 0.01, nd=2)
V.check('04.45', 'tab:fit_stability | protan folds favouring it', round(c9["gamma_folds_beat00"] * 7), 3, mode='eq')
V.check('04.46', 'tab:fit_stability | deutan Delta-L RDM atom', c8["rdm_dL_med"], -0.41, nd=2)
V.check('04.47', 'tab:fit_stability | deutan RDM folds (7 of 7)', round(c8["rdm_folds_beat00"] * 7), 7, mode='eq')
V.check('04.48', 'tab:fit_stability | protan Delta-L RDM atom', c9["rdm_dL_med"], -0.47, nd=2)
V.check('04.49', 'tab:fit_stability | protan RDM folds (7 of 7)', round(c9["rdm_folds_beat00"] * 7), 7, mode='eq')

{'deutan': [2.31, 1.937, 0.857, 2.188], 'protan': [0.805, -0.233, -0.476, -0.238]} 0.04600301659125189 0.08069381598793364
[OK ] 04.36 tab:fit_stability | deutan separation d at V1 / V2 / V3 / hV4: produced=(2.31, 1.94, 0.86, 2.19)  reported=(2.31, 1.94, 0.86, 2.19)
[OK ] 04.37 tab:fit_stability | protan separation d at V1 / V2 / V3 / hV4: produced=(0.81, -0.23, -0.48, -0.24)  reported=(0.81, -0.23, -0.48, -0.24)
[OK ] 04.38 Results §4 ¶1 | RDM atom admissible at all four ROIs in deutan and V1 alone in protan (d >= +0.5): produced=(4, (V1))  reported=(4, (V1))
[OK ] 04.39 tab:fit_stability | deutan grid percentile (%): produced=4.6  reported=4.6
[OK ] 04.40 tab:fit_stability | protan grid percentile (%): produced=8.069  reported=8.1
[OK ] 04.41 Results §4 ¶3 | both within the top 8% of the 1,326-cell grid: produced=True  reported=True
[OK ] 04.42 tab:fit_stability | deutan Delta-L psychophysical atom: produced=-13.85  reported=-13.85
[OK ] 04.43 tab:fit_stability | deutan folds favouri

### Neural-term ablation and resample stability (Results section 5; Supplementary tab:fit_stability)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.50 | Results §5 ¶1 | protan psychophysical atoms alone | `(26.0, 4.0)` |
| 04.51 | Results §5 ¶1 | protan RDM atom alone | `(0.0, 24.0)` |
| 04.52 | Results §5 ¶2 | deutan psychophysical atoms alone | `(16.0, -44.0)` |
| 04.53 | Results §5 ¶2 | deutan RDM atom alone | `(4.0, -26.0)` |
| 04.54 | Results §5 ¶2 | all three deutan argmins on the same side of the confusion axis (beta_c < 0) | `True` |
| 04.55 | tab:fit_stability | deutan selected combination, SRM basis | `(8.0, -42.0)` |
| 04.56 | tab:fit_stability | protan selected combination, SRM basis | `(32.0, 0.0)` |
| 04.57 | tab:fit_stability | deutan boundary-saturation rate, psychophysical alone | `0.23` |
| 04.58 | tab:fit_stability | deutan boundary-saturation rate, selected | `0.09` |
| 04.59 | Results §5 ¶2 | adding the RDM atom more than halved the deutan saturation rate | `True` |
| 04.60 | tab:fit_stability | protan boundary-saturation rate, psychophysical alone -> selected | `(0.0, 0.0)` |
| 04.61 | tab:fit_stability | deutan parameter IQR, PCA: psychophysical alone | `(18.0, 6.0)` |
| 04.62 | tab:fit_stability | deutan parameter IQR, PCA: selected | `(8.0, 2.0)` |
| 04.63 | tab:fit_stability | protan parameter IQR, PCA: psychophysical alone | `(6.0, 4.0)` |
| 04.64 | tab:fit_stability | protan parameter IQR, PCA: selected | `(0.0, 0.0)` |
| 04.65 | tab:fit_stability | deutan parameter IQR, SRM: psychophysical alone | `(18.0, 6.0)` |
| 04.66 | tab:fit_stability | deutan parameter IQR, SRM: selected | `(10.0, 4.0)` |
| 04.67 | tab:fit_stability | protan parameter IQR, SRM: psychophysical alone | `(6.0, 4.0)` |
| 04.68 | tab:fit_stability | protan parameter IQR, SRM: selected | `(0.0, 2.0)` |

In [6]:
for k in S:
    print(k, "psych", argmin(S[k], PSY[k]), "rdm", argmin(S[k], RDM[k]), "sel", argmin(S[k], SEL[k]), "srm", argmin(M[k], SEL[k]),
          "boundary", two(S[k], PSY[k])["boundary_rate"], two(S[k], SEL[k])["boundary_rate"], "iqr", iqr(S[k], PSY[k]), iqr(S[k], SEL[k]), iqr(M[k], PSY[k]), iqr(M[k], SEL[k]))
V.check('04.50', 'Results §5 ¶1 | protan psychophysical atoms alone', argmin(S["protan"], PSY["protan"]), (26.0, 4.0), mode='eq')
V.check('04.51', 'Results §5 ¶1 | protan RDM atom alone', argmin(S["protan"], RDM["protan"]), (0.0, 24.0), mode='eq')
V.check('04.52', 'Results §5 ¶2 | deutan psychophysical atoms alone', argmin(S["deutan"], PSY["deutan"]), (16.0, -44.0), mode='eq')
V.check('04.53', 'Results §5 ¶2 | deutan RDM atom alone', argmin(S["deutan"], RDM["deutan"]), (4.0, -26.0), mode='eq')
V.check('04.54', 'Results §5 ¶2 | all three deutan argmins on the same side of the confusion axis (beta_c < 0)', all(argmin(S["deutan"], c)[1] < 0 for c in (PSY["deutan"], RDM["deutan"], SEL["deutan"])), True, mode='eq')
V.check('04.55', 'tab:fit_stability | deutan selected combination, SRM basis', argmin(M["deutan"], SEL["deutan"]), (8.0, -42.0), mode='eq')
V.check('04.56', 'tab:fit_stability | protan selected combination, SRM basis', argmin(M["protan"], SEL["protan"]), (32.0, 0.0), mode='eq')
V.check('04.57', 'tab:fit_stability | deutan boundary-saturation rate, psychophysical alone', two(S["deutan"], PSY["deutan"])["boundary_rate"], 0.23, nd=2)
V.check('04.58', 'tab:fit_stability | deutan boundary-saturation rate, selected', two(S["deutan"], SEL["deutan"])["boundary_rate"], 0.09, nd=2)
V.check('04.59', 'Results §5 ¶2 | adding the RDM atom more than halved the deutan saturation rate', two(S["deutan"], SEL["deutan"])["boundary_rate"] < 0.5 * two(S["deutan"], PSY["deutan"])["boundary_rate"], True, mode='eq')
V.check('04.60', 'tab:fit_stability | protan boundary-saturation rate, psychophysical alone -> selected', (two(S["protan"], PSY["protan"])["boundary_rate"], two(S["protan"], SEL["protan"])["boundary_rate"]), (0.0, 0.0), mode='pair', nd=2)
V.check('04.61', 'tab:fit_stability | deutan parameter IQR, PCA: psychophysical alone', iqr(S["deutan"], PSY["deutan"]), (18.0, 6.0), mode='eq')
V.check('04.62', 'tab:fit_stability | deutan parameter IQR, PCA: selected', iqr(S["deutan"], SEL["deutan"]), (8.0, 2.0), mode='eq')
V.check('04.63', 'tab:fit_stability | protan parameter IQR, PCA: psychophysical alone', iqr(S["protan"], PSY["protan"]), (6.0, 4.0), mode='eq')
V.check('04.64', 'tab:fit_stability | protan parameter IQR, PCA: selected', iqr(S["protan"], SEL["protan"]), (0.0, 0.0), mode='eq')
V.check('04.65', 'tab:fit_stability | deutan parameter IQR, SRM: psychophysical alone', iqr(M["deutan"], PSY["deutan"]), (18.0, 6.0), mode='eq')
V.check('04.66', 'tab:fit_stability | deutan parameter IQR, SRM: selected', iqr(M["deutan"], SEL["deutan"]), (10.0, 4.0), mode='eq')
V.check('04.67', 'tab:fit_stability | protan parameter IQR, SRM: psychophysical alone', iqr(M["protan"], PSY["protan"]), (6.0, 4.0), mode='eq')
V.check('04.68', 'tab:fit_stability | protan parameter IQR, SRM: selected', iqr(M["protan"], SEL["protan"]), (0.0, 2.0), mode='eq')

deutan psych (16.0, -44.0) rdm (4.0, -26.0) sel (6.0, -42.0) srm (8.0, -42.0) boundary 0.23 0.09333333333333334 iqr (18.0, 6.0) (8.0, 2.0) (18.0, 6.0) (10.0, 4.0)
protan psych (26.0, 4.0) rdm (0.0, 24.0) sel (2.0, 24.0) srm (32.0, 0.0) boundary 0.0 0.0 iqr (6.0, 4.0) (0.0, 0.0) (6.0, 4.0) (0.0, 2.0)
[OK ] 04.50 Results §5 ¶1 | protan psychophysical atoms alone: produced=(26, 4)  reported=(26, 4)
[OK ] 04.51 Results §5 ¶1 | protan RDM atom alone: produced=(0, 24)  reported=(0, 24)
[OK ] 04.52 Results §5 ¶2 | deutan psychophysical atoms alone: produced=(16, -44)  reported=(16, -44)
[OK ] 04.53 Results §5 ¶2 | deutan RDM atom alone: produced=(4, -26)  reported=(4, -26)
[OK ] 04.54 Results §5 ¶2 | all three deutan argmins on the same side of the confusion axis (beta_c < 0): produced=True  reported=True
[OK ] 04.55 tab:fit_stability | deutan selected combination, SRM basis: produced=(8, -42)  reported=(8, -42)
[OK ] 04.56 tab:fit_stability | protan selected combination, SRM basis: produced=

### Resample structure and basis caveat (Supplementary S12 'Basis caveat', S13 'Resample structure')
Per-resample two-component fits of the selected combination (300 rows kept in the slimmed files).

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.69 | S13 'Resample structure' | deutan beta_c at -42 or -44 in 202 of 300 resamples | `202` |
| 04.70 | S13 | deutan beta_c stays within [-48, -36] | `(-48.0, -36.0)` |
| 04.71 | S13 | deutan beta_s distributed from 0 to 14 | `(0.0, 14.0)` |
| 04.72 | S12 'Basis caveat' | protan SRM basis: modal argmin (32, 0) in 171 of 300 resamples | `171` |
| 04.73 | S12 'Basis caveat' | protan SRM basis: beta_c positive in 17% | `17.0` |
| 04.74 | S12 'Basis caveat' | and negative in 26% | `26.0` |
| 04.75 | S12 'Basis caveat' | deutan sign holds in 300 of 300 resamples under both bases | `(300, 300)` |
| 04.76 | Results §4 ¶4 | deutan |beta_c| = 42 exceeds the 26-degree recovery uncertainty; protan 24 lies within it | `(True, True)` |

In [7]:
rs8 = S["deutan"]["resamples_2comp"][SEL["deutan"]]; rs9m = M["protan"]["resamples_2comp"][SEL["protan"]]; rs8m = M["deutan"]["resamples_2comp"][SEL["deutan"]]
bc8 = np.array([r["beta_c"] for r in rs8]); bs8 = np.array([r["beta_s"] for r in rs8])
n_42_44 = int(np.isin(bc8, (-42.0, -44.0)).sum())
pairs9m = [(r["beta_s"], r["beta_c"]) for r in rs9m]
n_modal9m = pairs9m.count((32.0, 0.0)); pos9m = np.mean([p[1] > 0 for p in pairs9m]); neg9m = np.mean([p[1] < 0 for p in pairs9m])
neg8 = int((bc8 < 0).sum()); neg8m = int(sum(r["beta_c"] < 0 for r in rs8m))
print(n_42_44, bc8.min(), bc8.max(), bs8.min(), bs8.max(), n_modal9m, pos9m, neg9m, neg8, neg8m)
V.check('04.69', "S13 'Resample structure' | deutan beta_c at -42 or -44 in 202 of 300 resamples", n_42_44, 202, mode='eq')
V.check('04.70', 'S13 | deutan beta_c stays within [-48, -36]', (float(bc8.min()), float(bc8.max())), (-48.0, -36.0), mode='eq')
V.check('04.71', 'S13 | deutan beta_s distributed from 0 to 14', (float(bs8.min()), float(bs8.max())), (0.0, 14.0), mode='eq')
V.check('04.72', "S12 'Basis caveat' | protan SRM basis: modal argmin (32, 0) in 171 of 300 resamples", n_modal9m, 171, mode='eq')
V.check('04.73', "S12 'Basis caveat' | protan SRM basis: beta_c positive in 17%", pos9m * 100, 17.0, nd=0)
V.check('04.74', "S12 'Basis caveat' | and negative in 26%", neg9m * 100, 26.0, nd=0)
V.check('04.75', "S12 'Basis caveat' | deutan sign holds in 300 of 300 resamples under both bases", (neg8, neg8m), (300, 300), mode='eq')
V.check('04.76', 'Results §4 ¶4 | deutan |beta_c| = 42 exceeds the 26-degree recovery uncertainty; protan 24 lies within it', (abs(argmin(S["deutan"], SEL["deutan"])[1]) > 26, abs(argmin(S["protan"], SEL["protan"])[1]) <= 26), (True, True), mode='eq')

202 -48.0 -36.0 0.0 14.0 171 0.17333333333333334 0.25666666666666665 300 300
[OK ] 04.69 S13 'Resample structure' | deutan beta_c at -42 or -44 in 202 of 300 resamples: produced=202  reported=202
[OK ] 04.70 S13 | deutan beta_c stays within [-48, -36]: produced=(-48, -36)  reported=(-48, -36)
[OK ] 04.71 S13 | deutan beta_s distributed from 0 to 14: produced=(0, 14)  reported=(0, 14)
[OK ] 04.72 S12 'Basis caveat' | protan SRM basis: modal argmin (32, 0) in 171 of 300 resamples: produced=171  reported=171
[OK ] 04.73 S12 'Basis caveat' | protan SRM basis: beta_c positive in 17%: produced=17.33  reported=17
[OK ] 04.74 S12 'Basis caveat' | and negative in 26%: produced=25.67  reported=26
[OK ] 04.75 S12 'Basis caveat' | deutan sign holds in 300 of 300 resamples under both bases: produced=(300, 300)  reported=(300, 300)
[OK ] 04.76 Results §4 ¶4 | deutan |beta_c| = 42 exceeds the 26-degree recovery uncertainty; protan 24 lies within it: produced=(True, True)  reported=(True, True)


### Sign stability across the two preprocessing pipelines (Supplementary S13)
The selected combinations refitted on the head-motion-corrected amplitudes, everything else fixed.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.77 | S13 'Sign stability' | deutan beta_c median, primary | `-42.0` |
| 04.78 | S13 | deutan beta_c median, head-motion correction | `-46.0` |
| 04.79 | S13 | deutan fraction of negative resamples, primary | `1.0` |
| 04.80 | S13 | deutan fraction negative, head-motion correction | `0.947` |
| 04.81 | S13 | protan beta_c median, primary | `24.0` |
| 04.82 | S13 | protan beta_c median, head-motion correction | `-12.0` |
| 04.83 | S13 | protan negative resamples, head-motion correction (%) | `79.3` |
| 04.84 | S13 | protan negative resamples, primary (none) | `0.0` |
| 04.85 | S13 | deutan combined boundary fraction rises from 0.09 | `0.09` |
| 04.86 | S13 | to 0.72 | `0.72` |

In [8]:
bs3 = J("beta_sign_three_arms.json")["subjects"]
a8 = bs3["sub-08"]["arms"]; a9 = bs3["sub-09"]["arms"]
bnd8_base = a8["baseline"]["frac_beta_c_at_lower_edge"] + a8["baseline"]["frac_beta_c_at_upper_edge"] + a8["baseline"]["frac_beta_s_at_edge"]
bnd8_hmc = a8["hmc_v2"]["frac_beta_c_at_lower_edge"] + a8["hmc_v2"]["frac_beta_c_at_upper_edge"] + a8["hmc_v2"]["frac_beta_s_at_edge"]
print(a8["baseline"], a8["hmc_v2"], a9["baseline"], a9["hmc_v2"], bnd8_base, bnd8_hmc)
V.check('04.77', "S13 'Sign stability' | deutan beta_c median, primary", a8["baseline"]["beta_c_median"], -42.0, nd=0)
V.check('04.78', 'S13 | deutan beta_c median, head-motion correction', a8["hmc_v2"]["beta_c_median"], -46.0, nd=0)
V.check('04.79', 'S13 | deutan fraction of negative resamples, primary', a8["baseline"]["p_beta_c_negative"], 1.0, nd=3)
V.check('04.80', 'S13 | deutan fraction negative, head-motion correction', a8["hmc_v2"]["p_beta_c_negative"], 0.947, nd=3)
V.check('04.81', 'S13 | protan beta_c median, primary', a9["baseline"]["beta_c_median"], 24.0, nd=0)
V.check('04.82', 'S13 | protan beta_c median, head-motion correction', a9["hmc_v2"]["beta_c_median"], -12.0, nd=0)
V.check('04.83', 'S13 | protan negative resamples, head-motion correction (%)', a9["hmc_v2"]["p_beta_c_negative"] * 100, 79.3, nd=1)
V.check('04.84', 'S13 | protan negative resamples, primary (none)', a9["baseline"]["p_beta_c_negative"], 0.0, nd=3)
V.check('04.85', 'S13 | deutan combined boundary fraction rises from 0.09', bnd8_base, 0.09, nd=2)
V.check('04.86', 'S13 | to 0.72', bnd8_hmc, 0.72, nd=2)

{'combo': 'γOY|RDMV2|noLOCO', 'beta_c_median': -42.0, 'beta_s_median': 6.0, 'p_beta_c_negative': 1.0, 'frac_beta_c_at_lower_edge': 0.0, 'frac_beta_c_at_upper_edge': 0.0, 'frac_beta_s_at_edge': 0.093, 'train_loss_median': -2.8924} {'combo': 'γOY|RDMV2|noLOCO', 'beta_c_median': -46.0, 'beta_s_median': 20.0, 'p_beta_c_negative': 0.947, 'frac_beta_c_at_lower_edge': 0.283, 'frac_beta_c_at_upper_edge': 0.0, 'frac_beta_s_at_edge': 0.437, 'train_loss_median': -2.2335} {'combo': 'γALL|RDMV1|noLOCO', 'beta_c_median': 24.0, 'beta_s_median': 2.0, 'p_beta_c_negative': 0.0, 'frac_beta_c_at_lower_edge': 0.0, 'frac_beta_c_at_upper_edge': 0.0, 'frac_beta_s_at_edge': 0.0, 'train_loss_median': -1.6807} {'combo': 'γALL|RDMV1|noLOCO', 'beta_c_median': -12.0, 'beta_s_median': 24.0, 'p_beta_c_negative': 0.793, 'frac_beta_c_at_lower_edge': 0.0, 'frac_beta_c_at_upper_edge': 0.107, 'frac_beta_s_at_edge': 0.0, 'train_loss_median': -1.4824} 0.093 0.72
[OK ] 04.77 S13 'Sign stability' | deutan beta_c median, prima

### Figure S1
The loss landscape on the full control pool needs the C010 amplitudes, which are not distributed; the committed figure is `../figures/figS1_landscape.pdf`.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 04.87 | Figure S1 | loss landscape reconstruction on the seven-control pool | `figure committed; regeneration requires the amplitude arrays (scripts/viz_closure_ground_plot.py)` |

In [9]:
print((Path("..") / "figures" / "figS1_landscape.pdf").exists())
V.flag('04.87', 'Figure S1 | loss landscape reconstruction on the seven-control pool', 'figure committed; regeneration requires the amplitude arrays (scripts/viz_closure_ground_plot.py)', 'figure committed; regeneration requires the amplitude arrays (scripts/viz_closure_ground_plot.py)')

True
[-- ] 04.87 Figure S1 | loss landscape reconstruction on the seven-control pool: reported=figure committed; regeneration requires the amplitude arrays (scripts/viz_closure_ground_plot.py)  NO COMMITTED ARTIFACT: figure committed; regeneration requires the amplitude arrays (scripts/viz_closure_ground_plot.py)


In [10]:
V.summary()


=== 04_distortion_model: 86/86 numeric checks reproduced exactly; 0 within one unit of the last printed digit; 0 mismatch, 0 error, 1 pointer-only ===
